```
Weather: Create a CSV file containing new hourly weather observations
for the period immediately following the original dataset. Generate realistic
values for the existing attributes and add one new column, humidity, to simulate
schema evolution. The humidity column should contain realistic relative humidity
values (typically between 20% and 100%) represented as numeric percentages
```

* Generate synthetic data based on the already existing `weather.csv`

* We will use the first 7% of the old weather observations, and create new ones for 2025 with some random variations in each column

* We dont include "snwd", "snwd_source", "wpgt","wpgt_source" columns since they are completely null (on all rows)

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA_ROOT = Path(r"..\..\landing")

INPUT_CSV = DATA_ROOT / "weather" / "weather.csv"

OUTPUT_SYNTH = "weather_2025_synthetic.csv"

FRACTION = 0.07
SEED = 42

# Max variation (+/-) per column, added on top of the copied values

# We decided on these ranges based on the std/variance of each column
ADDITIVE = {"temp": 2.5, "rhum": 5, "wspd": 2, "pres": 2, "cldc": 0.5, "wdir": 15}

# Columns where most values are 0: scale by (1 +/- x) so zeros stay zero
MULTIPLICATIVE = {"prcp": 0.2}
# Per-row jitter as a share of the max variation (kept small so hours stay smooth)
JITTER = 0.1

# keep a constant seed to reproduce the same data each time
rng = np.random.default_rng(SEED)

* Read the the first 7% of the old data

In [2]:
df = pd.read_csv(INPUT_CSV)
df = df.drop(columns=["snwd", "snwd_source", "wpgt", "wpgt_source"])
df = df.sort_values(["year", "month", "day", "hour"]).reset_index(drop=True)

display(df.head())

n = round(len(df) * FRACTION)
f"Will take the first {n} records from the old dataset"

,year,month,day,hour,temp,temp_source,rhum,rhum_source,prcp,prcp_source,wdir,wdir_source,wspd,wspd_source,pres,pres_source,cldc,cldc_source,coco,coco_source
0,2024,1,1,0,6.1,isd_lite,47,isd_lite,0.0,dwd_mosmix,280,isd_lite,9.4,isd_lite,1015.8,isd_lite,8,isd_lite,3.0,dwd_mosmix
1,2024,1,1,1,6.1,isd_lite,45,isd_lite,0.0,isd_lite,260,isd_lite,13.0,isd_lite,1015.8,isd_lite,7,dwd_mosmix,3.0,dwd_mosmix
2,2024,1,1,2,6.1,isd_lite,47,isd_lite,0.0,isd_lite,250,isd_lite,13.0,isd_lite,1015.7,isd_lite,8,dwd_mosmix,3.0,dwd_mosmix
3,2024,1,1,3,6.1,isd_lite,51,isd_lite,0.0,isd_lite,260,isd_lite,13.0,isd_lite,1015.7,isd_lite,8,isd_lite,4.0,dwd_mosmix
4,2024,1,1,4,6.1,isd_lite,51,isd_lite,0.0,isd_lite,250,isd_lite,13.0,isd_lite,1015.7,isd_lite,8,dwd_mosmix,4.0,dwd_mosmix


'Will take the first 615 records from the old dataset'

In [3]:
df["cldc"]

0       8
1       7
2       8
3       8
4       8
       ..
8779    6
8780    7
8781    6
8782    8
8783    7
Name: cldc, Length: 8784, dtype: int64

* Mutate the newly generated data

In [4]:
new = df.head(n).copy()          # first 7% of 2024 (Jan 1 -> ~Jan 26)
new["year"] = 2025               # every 2024 date up to Jan 26 exists in 2025

# One random shift per column, applied to ALL rows -> consecutive hours stay
# consistent with each other, plus a tiny per-row jitter for variety.
for col, mag in ADDITIVE.items():
    shift = rng.uniform(-mag, mag)
    jitter = rng.uniform(-mag * JITTER, mag * JITTER, n)
    new[col] = new[col] + shift + jitter

for col, mag in MULTIPLICATIVE.items():
    new[col] = new[col] * (1 + rng.uniform(-mag, mag))

# Keep values physically sensible
new["wdir"] = new["wdir"] % 360
for col in [c for c in list(ADDITIVE) + list(MULTIPLICATIVE) if c != "wdir"]:
    new[col] = new[col].clip(df[col].min(), df[col].max())
new["rhum"] = new["rhum"].clip(0, 100)
# new["wpgt"] = new["wpgt"].clip(lower=new["wspd"])   # gusts >= mean wind speed

new[list(ADDITIVE) + list(MULTIPLICATIVE)] = new[list(ADDITIVE) + list(MULTIPLICATIVE)].round(1)

# rhumn, wdir and cldc are integers, not floats
new[["rhum", "wdir", "cldc"]] = new[["rhum", "wdir", "cldc"]].round(0).astype(int)

# Mark the rows as fake
for col in [c for c in new.columns if c.endswith("_source")]:
    new[col] = "synthetic"

* Add a new `humidity` column (uniformly distributed between 0.2 and 1)

In [5]:
new["humidity"] = rng.uniform(20, 100, size=len(new)).round(1) # we use (20, 100) instead of (0.2, 1) because 'rhum' also follows this format
new["humidity"].head()

0    23.8
1    22.0
2    54.4
3    90.7
4    39.5
Name: humidity, dtype: float64

* Make sure the types are the same (e.g. we dont use floats on integer columns on the new data)

In [6]:
df.dtypes

year             int64
month            int64
day              int64
hour             int64
temp           float64
temp_source     object
rhum             int64
rhum_source     object
prcp           float64
prcp_source     object
wdir             int64
wdir_source     object
wspd           float64
wspd_source     object
pres           float64
pres_source     object
cldc             int64
cldc_source     object
coco           float64
coco_source     object
dtype: object

In [7]:
new.dtypes

year             int64
month            int64
day              int64
hour             int64
temp           float64
temp_source     object
rhum             int64
rhum_source     object
prcp           float64
prcp_source     object
wdir             int64
wdir_source     object
wspd           float64
wspd_source     object
pres           float64
pres_source     object
cldc             int64
cldc_source     object
coco           float64
coco_source     object
humidity       float64
dtype: object

* Store the new dataset

In [8]:
new.to_csv(OUTPUT_SYNTH, index=False)
print(f"Saved {len(new)} new synthetic records to {OUTPUT_SYNTH}")

Saved 615 new synthetic records to weather_2025_synthetic.csv
